<a href="https://colab.research.google.com/github/inhyuk78/Re-implementation-of-MOLI-By-Hossein-Sharifi-Noghabi-/blob/main/04_Drug_Response_Prediction_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from itertools import combinations
from scipy import stats

## **Importing X_train, X_test, y_train, y_test data**

In [24]:
data = torch.load('/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/multi_omics.pt')

X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']

## **Train/Test split of multi-omics tensor data**

In [25]:
X_train = X_train.float()
y_train = y_train.view(-1).float()

X_test = X_test.float()
y_test = y_test.view(-1).float()

## **NN model architecture**

In [26]:
class MOLIClassifier(nn.Module):
  '''
  Classifier for MOLI (Multi-Omics Late Integration)

  Takes a concatenated multi-omics feature vector (eg. RNA-seq + mutation + CNA), runs it through low-dimensional embedding layer before
  producing binary prediction (ie. drug response).

  Architecture:
    input_dim -> Linear(512) -> BatchNorm -> ReLU -> Dropout(0.3)
              -> Linear(32)  -> BatchNorm -> ReLU -> Dropout(0.3) [embedding]
              -> Linear(1)   -> Sigmoid                           [prediction]

  Args:
    input_dim (int): Dimensionality of input multi-omics feature vector (eg. 768-features for concatenated RNA-seq, mutation, CNA encodings).

  Returns:
    pred (torch.Tensor): Predicted probabilities of shape (batch_size, ).
    embedding (torch.Tensor): 32-dim embdding of shape (batch_size, 32) used for Triplet loss mining.
  '''
  def __init__(self, input_dim):
    super().__init__()

    self.feature = nn.Sequential(
        nn.Linear(input_dim, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.3)
    )

    self.embed = nn.Sequential(
        nn.Linear(512, 32),
        nn.BatchNorm1d(32),
        nn.ReLU(),
        nn.Dropout(0.3)
    )

    self.fc_out = nn.Linear(32, 1)     # Single logit output for BCELoss

  def forward(self,x):
    z = self.feature(x)
    embedding = self.embed(z)
    pred = torch.sigmoid(self.fc_out(embedding)).squeeze(1)
    return pred, embedding

## **Triplet Mining**

In [27]:
def mine_all_triplets(embedding, labels):
  '''
  Mine all triplets from a batch for triplet loss training.

  Performs (all-combinations) triplet mining within a batch: for every same-class pair (ie. anchor, positive), every opposite-class sample
  is paired as a negative. This is done in the same manner for both classes (drug-sensitive, drug-resistant), so returned triplets pull
  same-class embeddings together and push opposite-class embeddings apart in euclidean space.

  Args:
    embedding (torch.Tensor): Batch of embeddings of shape (batch_size, embed_dim), output of MOLIClassifier's embedding layer.
    labels (torch.Tensor): Binary labels of shape (batch_size,), where 1 = sensitive and 0 = resistant.

  Returns:
    anchors (torch.Tensor or None): Stacked anchor embeddings of shape (num_triplets, embed_dim).
    positives (torch.Tensor or None): Stacked positive embeddings of shape (num_triplets, embed_dim), same class as anchor.
    negatives (torch.Tensor or None): Stacked negative embeddings of shape (num_triplets, embed_dim), opposite class from anchor.

    Returns (None, None, None) if batch does not contain at least 2 samples of any one class (ie. no valid anchor-positive pair can be formed).
  '''
  anchors, positives, negatives = [], [], []

  sensitive_indices = torch.where(labels==1)[0]
  resistant_indices = torch.where(labels==0)[0]

  # Drug-sensitive (label=1): Anchor = Sensitive | Positive = Sensitive | Negative = Resistant
  for anchor_indices, positive_indices in combinations(sensitive_indices.tolist(), 2):   # pull unique pair combinations of anchor/positive
    for negative_indices in resistant_indices.tolist():  # pull single negative
      anchors.append(embedding[anchor_indices])
      positives.append(embedding[positive_indices])
      negatives.append(embedding[negative_indices])

  # Drug-resistant (label=0): Anchor = Resistant | Positive = Resistant | Negative = Sensitive
  for anchor_indices, positive_indices in combinations(resistant_indices.tolist(), 2):
    for negative_indices in sensitive_indices.tolist():
      anchors.append(embedding[anchor_indices])
      positives.append(embedding[positive_indices])
      negatives.append(embedding[negative_indices])

  if len(anchors) == 0:
    return None, None, None

  return (
      torch.stack(anchors),
      torch.stack(positives),
      torch.stack(negatives)
  )

## **Combined Loss**

In [28]:
# Combined loss: J = BCELoss + TripletLoss
bce_loss = nn.BCELoss()
triplet_loss = nn.TripletMarginLoss(margin=1.0, p=2)

# Enforces d(anchor, negative) - d(anchor, positive) >= margin, penalizing violations proportionally.
# p=2 -> Euclidean distance.

In [29]:
def combined_loss(preds, labels, anchors, positives, negatives, gamma=0.1):
  '''
  Combined loss for MOLI training: BCELoss + weighted TripletLoss

  total_loss = BCELoss + (gamma * TripletLoss)

  Args:
    preds (torch.Tensor): Predicted probabilities of shape (batch_size,), output of MOLIClassifier.
    labels (torch.Tensor): Binary labels of shape (batch_size,)
    anchors (torch.Tensor or None): Anchor embeddings of shape (num_triplets, embed_dim) from mine_all_triplets.
    positives (torch.Tensor or None): Positive embeddings of shape (num_triplets, embed_dim) from mine_all_triplets.
    negatives (torch.Tensor or None): Negative embeddings of shape (num_triplets, embed_dim) from mine_all_triplets.
    gamma (float, default=0.1): weight applied to triplet loss.

  Returns:
    total_loss (torch.Tensor): Combined loss for backpropagation.
    pred_loss (torch.Tensor): BCELoss component (unweighted).
    trip_loss (torch.Tensor): TripletLoss component (unweighted).
  '''
  pred_loss = bce_loss(preds, labels.float())

  if anchors is None:
    trip_loss = torch.tensor(0.0, requires_grad=True)
  else:
    trip_loss = triplet_loss(anchors, positives, negatives)

  total_loss = pred_loss + (gamma * trip_loss)
  return total_loss, pred_loss, trip_loss

In [30]:
dataset = TensorDataset(X_train, y_train)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [31]:
def train_model(model, dataloader, optimizer, epochs):
  '''
  Train MOLIClassifier using combined BCELoss + TripletLoss

  For each batch: runs forward pass to get predictions and embeddings, mines triplets from embeddings, computes combined loss,
  and updated model weights via backpropagation. Logs average total, BCE, triplet loss per epoch.

  Args:
    model (nn.Module): MOLIClassifier to train.
    dataloader (DataLoader): Yields (batch_x, batch_y) batches of multi-omics features and binary labels.
    optimizer (torch.optim.Optimizer): Optimizer used to update model weights.
    epochs (int): Number of training epochs.

  Returns:
    None. Trains model in-place and prints per-epoch loss summary.
  '''
  model.train()

  for epoch in range(epochs):
    epoch_loss = 0
    epoch_pred_loss = 0
    epoch_trip_loss = 0

    for batch_x, batch_y in dataloader:
      optimizer.zero_grad()
      preds, embeddings = model(batch_x) # Forward pass
      anchors, positives, negatives = mine_all_triplets(embeddings, batch_y) # Mine triplets
      total_loss, pred_loss, trip_loss = combined_loss(preds, batch_y, anchors, positives, negatives) # Total loss
      total_loss.backward() # Backpropagation
      optimizer.step() # Update weights

      epoch_loss += total_loss.item()
      epoch_pred_loss += pred_loss.item()
      epoch_trip_loss += trip_loss.item()

    n_batches = len(dataloader)
    print(
        f'Epoch [{epoch+1}/{epochs}] | '
        f'Total: {epoch_loss/n_batches} | '
        f'BCE: {epoch_pred_loss/n_batches} | '
        f'Triplet: {epoch_trip_loss/n_batches} '
    )

## **Training & Evaluating Model**

In [32]:
def evaluate_model(model, dataloader):
  '''
  Evaluate a trained model on a dataset and compute AUC and accuracy.

  Runs the model over all batches in the dataloader, collects predictions and true labels, then computes ROC-AUC and binary classification accuracy
  using a 0.5 decision threshold.

  Args:
    model (torch.nn.Module): Trained model
    dataloader (torch.utils.data.DataLoader): Yields batches of (batch_x, batch_y), where batch_y contains binary 'true' labels.

  Returns:
    auc (float): Area under the ROC curve.
    accuracy (float): Proportion of samples correctly classified using a 0.5 threshold.
  '''
  model.eval()

  all_preds = []
  all_labels = []

  with torch.no_grad():
    for batch_x, batch_y in dataloader:
      preds, _ = model(batch_x)
      all_preds.append(preds)
      all_labels.append(batch_y)

  all_preds = torch.cat(all_preds).numpy()
  all_labels = torch.cat(all_labels).numpy()

  auc = roc_auc_score(all_labels, all_preds)
  binary_preds = (all_preds >= 0.5).astype(int)
  accuracy = (binary_preds == all_labels).mean()
  return auc, accuracy

In [33]:
def run_variant(train_fn, seed=42, epochs=20):
  '''
  Train a fresh MOLIClassifier using the given training function and evaluate it.

  Instantiates a new model and optimizer from scratch (this is important since the Adam optimizer holds internal state
  that must not persist across variants), trains it using train_fn, then evaluates it using evaluate_model function.

  Args:
    train_fn (callable): Training function with distinct train_fn(model, dataloader, optimizer, epochs) (eg. train_model_no_trip_loss()).
    seed (int, default=42): Random seed for reproducibility.
    epochs (int, default=20): Number of training epochs.

  Returns:
    tuple (auc, accuracy): From evaluate_model on test set.
  '''
  torch.manual_seed(seed)
  model = MOLIClassifier(input_dim=768)
  optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
  train_fn(model, dataloader, optimizer, epochs=epochs)
  return evaluate_model(model,test_dataloader)

## **<u>Ablation Study</u>:**
## **Comparing Triplet Loss, Contrastive Loss, and BCE-only training.**

To evaluate whether MOLI's triplet loss is necessary for performance, we compare three training variants:

#### **BCE-only (baseline):** No embedding loss - trains purely on binary cross-entropy for drug response prediction.
#### **Triplet loss (as in original paper):** Operates on triplets (anchor, positive, negative). Pulls positive embedding closer to anchor, and pushes the negative further away.
#### **Contrastive loss:** Operates on pairs. Pulls similar pairs together and pushes dissimilar pairs apart in embedding space.

#### 1) No Triplet Loss (pure classification loss)

In [34]:
def train_model_no_trip_loss(model, dataloader, optimizer, epochs):
  '''
  Train MOLIClassifier using BCE loss only (no triplet loss).

  Args:
    model (torch.nn.Module): MOLIClassifier instance to train.
    dataloader (torch.utils.data.DataLoader): Yields batches of (batch_x, batch_y) for training.
    optimizer (torch.optim.Optimizer): Optimizer for updating model parameters.
    epochs (int): Number of training epochs.

  Returns:
    None. Model is trained in place and prints per-epoch loss summary.
  '''
  model.train()

  for epoch in range(epochs):
    epoch_loss = 0
    epoch_pred_loss = 0

    for batch_x, batch_y in dataloader:
      optimizer.zero_grad()
      preds, _ = model(batch_x)
      pred_loss = bce_loss(preds, batch_y.float())
      pred_loss.backward()
      optimizer.step()

      epoch_loss += pred_loss.item()
      epoch_pred_loss += pred_loss.item()

    n_batches = len(dataloader)
    print(
        f'Epoch: [{epoch+1}/{epochs}] | '
        f'Total: {epoch_loss/n_batches} | '
        f'BCE: {epoch_pred_loss/n_batches}'
    )

#### 2) (Pairwise) Contrastive Loss

In [35]:
class ContrastiveLoss(nn.Module):
  '''
  Contrastive loss for learning an embedding space where similar pairs are pulled together and dissimilar pairs
  are pushed apart. Used as alternative to triplet loss in the loss ablation study.
  '''
  def __init__(self,margin=1.0):
    '''
    Args:
      margin (float, default=1.0): Minimum desired distance between dissimilar pairs.
    '''
    super().__init__()
    self.margin = margin

  def forward(self,emb1,emb2,label):
    '''
    Compute contrastive loss for a batch of embedding pairs.

    Args:
      emb1 (torch.Tensor): First embedding in each pair.
      emb2 (torch.Tensor): Second embedding in each pair.
      label (torch.Tensor): 1 if (emb1, emb2) is similar (ie. same drug response), 0 if dissimilar.

    Returns:
      torch.Tensor: Mean contrastive loss over the batch.
    '''
    distance = F.pairwise_distance(emb1, emb2)  # Euclidean distance

    # relu (max(0,x)) clips negative values since 'negative loss' is not meaningful
    loss = label * distance.pow(2) + (1 - label) * F.relu(self.margin - distance).pow(2)
    return loss.mean()

In [36]:
def mine_all_pairs(embedding, labels):
  '''
  Construct all possible pairs of embeddings for contrastive loss training.

  Args:
    embedding (torch.Tensor): Batch of embeddings of shape (batch_size, embed_dim), output of MOLIClassifier's embedding layer.
    labels (torch.Tensor): Binary drug response labels, shape (batch_size,) , where 1 = sensitive and 0 = resistant.

  Returns:
    tuple:
      emb1 (torch.Tensor or None): First embedding in each pair, shape (n_pairs, embed_dim).
      emb2 (torch.Tensor or None): Second embedding in each pair, shape (n_pairs, embed_dim).
      pair_labels (torch.Tensor or None): 1 if pair is same class, 0 if different class, shape (n_pairs,).
    Returns (None, None, None) if no pairs can be formed (eg. batch contains only 1 class).
  '''
  emb1_list, emb2_list, pair_labels = [], [], []

  sensitive_indices = torch.where(labels==1)[0]
  resistant_indices = torch.where(labels==0)[0]

  # Label: 1 (sensitive-sensitive pairs)
  for i, j in combinations(sensitive_indices.tolist(),2):
    emb1_list.append(embedding[i])
    emb2_list.append(embedding[j])
    pair_labels.append(1)

  # Label: 1 (resistant-resistant pairs)
  for i, j in combinations(resistant_indices.tolist(),2):
    emb1_list.append(embedding[i])
    emb2_list.append(embedding[j])
    pair_labels.append(1)

  # Label: 0 (sensitive-resistant pairs)
  for i in sensitive_indices.tolist():
    for j in resistant_indices.tolist():
      emb1_list.append(embedding[i])
      emb2_list.append(embedding[j])
      pair_labels.append(0)

  if len(emb1_list) == 0:
    return None, None, None

  return (
      torch.stack(emb1_list),
      torch.stack(emb2_list),
      torch.tensor(pair_labels, dtype=torch.float32, device=embedding.device)
  )

In [37]:
contrastive_loss_fn = ContrastiveLoss(margin=1.0)

def combined_loss_contrastive(preds, labels, emb1, emb2, pair_labels, gamma=0.1):
  '''
  Combined loss: BCELoss + weighted ContrastiveLoss.

  total_loss = BCELoss + (gamma * ContrastiveLoss).

  Args:
    preds (torch.Tensor): Predicted probabilities of shape (batch_size,), output of MOLIClassifier.
    labels (torch.Tensor): Binary labels of shape (batch_size,)
    emb1 (torch.Tensor or None): First embedding in each mined pair, shape (n_pairs, embed_dim). None if no pairs formed.
    emb2 (torch.Tensor or None): Second embedding in each mined pair, shape (n_pairs, embed_dim). None if no pairs formed.
    pair_labels (torch.Tensor or None): 1 if pair is same class, 0 if different class, shape (n_pairs,).
    gamma (float, default=0.1): Weight applied to contrastive loss.

  Returns:
    total_loss (torch.Tensor): Combined loss for backpropagation.
    pred_loss (torch.Tensor): BCELoss component (unweighted).
    contras_loss (torch.Tensor): ContrastiveLoss component (unweighted).
  '''
  pred_loss = bce_loss(preds, labels.float()) # BCELoss

  if emb1 is None:
    contras_loss = torch.tensor(0.0, requires_grad=True, device=preds.device) # ContrastiveLoss
  else:
    contras_loss = contrastive_loss_fn(emb1, emb2, pair_labels)

  total_loss = pred_loss + (gamma * contras_loss)
  return total_loss, pred_loss, contras_loss

In [38]:
def train_model_contrastive_loss(model, dataloader, optimizer, epochs):
  '''
  Train MOLIClassifier using combined BCELoss + ContrastiveLoss.

  Args:
    model (torch.nn.Module): MOLIClassifier instance to train.
    dataloader (torch.utils.data.DataLoader): Yields batches of (batch_x, batch_y) for training.
    optimizer (torch.optim.Optimizer): Optimizer for updating model parameters.
    epochs (int): Number of training epochs.

  Returns:
    None. Model is trained in place and prints per-epoch loss summary.
  '''
  model.train()

  for epoch in range(epochs):
    epoch_loss = 0
    epoch_pred_loss = 0
    epoch_contrastive_loss = 0

    for batch_x, batch_y in dataloader:
      optimizer.zero_grad()
      preds, embeddings = model(batch_x) # Forward pass
      emb1, emb2, pair_labels = mine_all_pairs(embeddings, batch_y) # Mine pairs
      total_loss, pred_loss, contrastive_loss = combined_loss_contrastive(preds, batch_y, emb1, emb2, pair_labels) # Total loss
      total_loss.backward() # Backpropagation
      optimizer.step() # Update weights

      epoch_loss += total_loss.item()
      epoch_pred_loss += pred_loss.item()
      epoch_contrastive_loss += contrastive_loss.item()

    n_batches = len(dataloader)
    print(
        f'Epoch [{epoch+1}/{epochs}] | ',
        f'Total: {epoch_loss/n_batches} | ',
        f'BCE: {epoch_pred_loss/n_batches} | ',
        f'Contrastive: {epoch_contrastive_loss/n_batches}'
    )

In [39]:
auc_triplet, acc_triplet = run_variant(train_model)
auc_bce, acc_bce = run_variant(train_model_no_trip_loss)
auc_contrastive, acc_contrastive = run_variant(train_model_contrastive_loss)

print()
print('_________________________________________________________________________')
print(f'AUC, accuracy of model with triplet loss: {auc_triplet} | {acc_triplet}')
print('_________________________________________________________________________')
print(f'AUC, accuracy of model with NO triplet loss: {auc_bce} | {acc_bce}')
print('_________________________________________________________________________')
print(f'AUC, accuracy of model with contrastive loss: {auc_contrastive} | {acc_contrastive}')

Epoch [1/20] | Total: 0.8011749386787415 | BCE: 0.6628908731720664 | Triplet: 1.3828405792062932 
Epoch [2/20] | Total: 0.717336735942147 | BCE: 0.603544598275965 | Triplet: 1.1379214308478616 
Epoch [3/20] | Total: 0.708464882590554 | BCE: 0.6040605469183489 | Triplet: 1.044043410908092 
Epoch [4/20] | Total: 0.6785959167913957 | BCE: 0.5752272443337874 | Triplet: 1.033686719157479 
Epoch [5/20] | Total: 0.667500376701355 | BCE: 0.5638078288598494 | Triplet: 1.0369255434383045 
Epoch [6/20] | Total: 0.6580280553210865 | BCE: 0.5555376356298273 | Triplet: 1.0249040831219067 
Epoch [7/20] | Total: 0.6409661065448414 | BCE: 0.5409615608778867 | Triplet: 1.0000455379486084 
Epoch [8/20] | Total: 0.6393821347843517 | BCE: 0.5409179844639518 | Triplet: 0.9846415302970193 
Epoch [9/20] | Total: 0.6311742771755565 | BCE: 0.5341549597003243 | Triplet: 0.9701930934732611 
Epoch [10/20] | Total: 0.6283712387084961 | BCE: 0.5281010974537242 | Triplet: 1.0027014829895713 
Epoch [11/20] | Total: 0.

###**Multi-seed comparison**

In [40]:
results = {
    'triplet': [],
    'no_triplet': [],
    'contrastive': []
}

In [20]:
for seed in range(5):
  results['triplet'].append(run_variant(train_model, seed=seed))
  results['no_triplet'].append(run_variant(train_model_no_trip_loss, seed=seed))
  results['contrastive'].append(run_variant(train_model_contrastive_loss, seed=seed))

for variant, runs in results.items():
  aucs = []

  for r in runs:
    aucs.append(r[0])

  print()
  print(f'{variant}: AUC {np.mean(aucs):.3f} ± {np.std(aucs):.3f}')

Epoch [1/20] | Total: 0.8038113821636547 | BCE: 0.6590070399371061 | Triplet: 1.448043335567821 
Epoch [2/20] | Total: 0.719932805408131 | BCE: 0.5998475009744818 | Triplet: 1.2008529576388272 
Epoch [3/20] | Total: 0.6963048577308655 | BCE: 0.5807758380066265 | Triplet: 1.1552902243354104 
Epoch [4/20] | Total: 0.6776702349836176 | BCE: 0.5700038183819164 | Triplet: 1.0766641931100325 
Epoch [5/20] | Total: 0.6556849371303212 | BCE: 0.5527982115745544 | Triplet: 1.0288673530925403 
Epoch [6/20] | Total: 0.6420786109837618 | BCE: 0.5442795347083699 | Triplet: 0.9779907410795038 
Epoch [7/20] | Total: 0.6550612503832037 | BCE: 0.5487145781517029 | Triplet: 1.063466733152216 
Epoch [8/20] | Total: 0.6296132586219094 | BCE: 0.5325801778923381 | Triplet: 0.9703307531096719 
Epoch [9/20] | Total: 0.6328898072242737 | BCE: 0.5351947302168066 | Triplet: 0.9769508242607117 
Epoch [10/20] | Total: 0.6099538857286627 | BCE: 0.5131879150867462 | Triplet: 0.9676597226749767 
Epoch [11/20] | Total:

In [21]:
aucs = {}

for variant, runs in results.items():
  auc_list = []

  for r in runs:
    auc_list.append(r[0])

  aucs[variant] = np.array(auc_list)


# Per-seed AUCs
print("seed  triplet  bce  contrastive")
for i in range(5):
  print(f"{i}  {aucs['triplet'][i]:.3f}  {aucs['no_triplet'][i]:.3f}  {aucs['contrastive'][i]:.3f}")

# Mean ± sample SD (ddof=1)
print()
for k, v in aucs.items():
  print(f"{k}:  {v.mean():.3f}  ± {v.std(ddof=1):.3f}")

# Paired t-tests
print()
pairs = [("BCE vs Triplet",         "no_triplet", "triplet"),
         ("BCE vs Contrastive",     "no_triplet", "contrastive"),
         ("Triplet vs Contrastive", "triplet", "contrastive")]

for name, a, b in pairs:
  t, p = stats.ttest_rel(aucs[a], aucs[b])
  d = aucs[a] - aucs[b]
  print(f"{name}: AUC difference = {d.mean():+.4f}, t = {t:.2f}, p = {p:.4f}")

seed  triplet  bce  contrastive
0  0.702  0.695  0.683
1  0.694  0.708  0.695
2  0.708  0.711  0.689
3  0.696  0.714  0.686
4  0.680  0.704  0.695

triplet:  0.696  ± 0.011
no_triplet:  0.706  ± 0.007
contrastive:  0.690  ± 0.006

BCE vs Triplet: AUC difference = +0.0101, t = 1.81, p = 0.1441
BCE vs Contrastive: AUC difference = +0.0167, t = 4.59, p = 0.0101
Triplet vs Contrastive: AUC difference = +0.0066, t = 1.00, p = 0.3738
